In [1]:
import math                                                    # pour sqrt et les calculs mathématiques

# Dataset : chaque ligne = un jour météo [Temp(°C), Humidité(%), Vent(km/h), Pression(hPa)]
data = [
    [35, 90, 10, 1008],                                        # jour 1
    [33, 85, 25, 1010],                                        # jour 2
    [28, 70,  5, 1015],                                        # jour 3
    [20, 60, 15, 1020],                                        # jour 4
    [15, 50, 30, 1025],                                        # jour 5
    [18, 55, 20, 1022],                                        # jour 6
    [30, 80, 10, 1012],                                        # jour 7
    [25, 65, 12, 1018],                                        # jour 8
    [10, 45, 35, 1030],                                        # jour 9
    [22, 62, 18, 1019],                                        # jour 10
]

# ──────────────────────────────────────────────
# ÉTAPE 1 : Centrer les données (soustraire la moyenne)
# PCA a besoin que les données soient centrées sur 0
# ──────────────────────────────────────────────
def centrer(data):
    n_cols = len(data[0])                                      # nombre de colonnes (ici 4)
    moyennes = []                                              # liste pour stocker la moyenne de chaque colonne

    for col in range(n_cols):                                  # on parcourt chaque colonne
        moy = sum(row[col] for row in data) / len(data)        # moyenne = somme / nombre de lignes
        moyennes.append(moy)                                   # on sauvegarde la moyenne de cette colonne

    data_centree = []                                          # liste pour les données après centrage
    for row in data:                                           # pour chaque ligne du dataset
        ligne_centree = [row[col] - moyennes[col]              # soustraire la moyenne de chaque valeur
                         for col in range(n_cols)]
        data_centree.append(ligne_centree)                     # ajouter la ligne centrée au résultat

    return data_centree, moyennes                              # retourner les données centrées + les moyennes

# ──────────────────────────────────────────────
# ÉTAPE 2 : Calculer la matrice de covariance
# covariance(i,j) = à quel point les colonnes i et j varient ensemble
# ──────────────────────────────────────────────
def matrice_covariance(data):
    n      = len(data)                                         # nombre de lignes
    n_cols = len(data[0])                                      # nombre de colonnes

    cov = []                                                   # la matrice de covariance (n_cols x n_cols)
    for i in range(n_cols):                                    # pour chaque colonne i
        ligne = []                                             # une ligne de la matrice
        for j in range(n_cols):                                # pour chaque colonne j
            cov_ij = sum(data[k][i] * data[k][j]              # multiplier les valeurs ligne par ligne
                         for k in range(n)) / (n - 1)         # diviser par (n-1) : formule standard
            ligne.append(cov_ij)                               # ajouter la valeur cov(i,j) à la ligne
        cov.append(ligne)                                      # ajouter la ligne à la matrice

    return cov                                                 # retourner la matrice complète

# ──────────────────────────────────────────────
# ÉTAPE 3 : Afficher une matrice proprement
# ──────────────────────────────────────────────
def afficher_matrice(M, titre="Matrice"):
    print(f"\n── {titre} ──")                                  # afficher le titre
    for ligne in M:                                            # pour chaque ligne de la matrice
        print("  " + "  ".join(f"{v:10.2f}" for v in ligne))  # afficher les valeurs alignées

# ──────────────────────────────────────────────
# ÉTAPE 4 : Outils mathématiques pour les vecteurs
# ──────────────────────────────────────────────
def norme(v):
    return math.sqrt(sum(x**2 for x in v))                    # longueur du vecteur = racine(somme des carrés)

def normaliser(v):
    n = norme(v)                                               # calculer la longueur du vecteur
    return [x / n for x in v]                                 # diviser chaque composante → vecteur de longueur 1

def produit_matrice_vecteur(M, v):
    return [sum(M[i][j] * v[j]                                # multiplier chaque ligne de M par v
                for j in range(len(v)))
            for i in range(len(M))]                           # retourne un nouveau vecteur

# ──────────────────────────────────────────────
# ÉTAPE 5 : Trouver valeurs propres et vecteurs propres
# Méthode : Power Iteration (itération de la puissance)
# → répéter M×v jusqu'à convergence vers la direction dominante
# ──────────────────────────────────────────────
def power_iteration(M, n_iter=1000):
    n    = len(M)                                              # taille de la matrice
    vect = [1.0] * n                                          # vecteur de départ : tous les 1

    for _ in range(n_iter):                                    # répéter 1000 fois
        vect = produit_matrice_vecteur(M, vect)               # appliquer la matrice au vecteur
        vect = normaliser(vect)                               # normaliser pour éviter les grands nombres

    Mv = produit_matrice_vecteur(M, vect)                     # calculer M × vect une dernière fois
    valeur_propre = sum(vect[i] * Mv[i] for i in range(n))   # valeur propre = v^T × M × v

    return valeur_propre, vect                                 # retourner (valeur propre, vecteur propre)

# ──────────────────────────────────────────────
# ÉTAPE 6 : Déflation — retirer la contribution d'un composant trouvé
# pour pouvoir trouver le composant suivant indépendamment
# ──────────────────────────────────────────────
def deflater(M, valeur_propre, vecteur):
    n  = len(M)                                                # taille de la matrice
    M2 = [[0.0] * n for _ in range(n)]                        # nouvelle matrice vide

    for i in range(n):                                         # pour chaque ligne
        for j in range(n):                                     # pour chaque colonne
            M2[i][j] = M[i][j] - valeur_propre * vecteur[i] * vecteur[j]  # soustraire la contribution du composant

    return M2                                                  # retourner la matrice réduite

# ──────────────────────────────────────────────
# ÉTAPE 7 : Trouver les N premiers composants principaux
# chaque composant = direction de variance maximale dans les données
# ──────────────────────────────────────────────
def composants_principaux(cov, n_composants):
    M          = [ligne[:] for ligne in cov]                   # copie de la matrice de covariance
    composants = []                                            # liste des vecteurs propres (directions)
    valeurs    = []                                            # liste des valeurs propres (importances)

    for _ in range(n_composants):                              # répéter pour chaque composant voulu
        val, vect = power_iteration(M)                         # trouver le composant le plus fort restant
        composants.append(vect)                                # sauvegarder le vecteur propre
        valeurs.append(val)                                    # sauvegarder la valeur propre
        M = deflater(M, val, vect)                             # retirer ce composant pour trouver le suivant

    return valeurs, composants                                 # retourner toutes les valeurs et directions

# ──────────────────────────────────────────────
# ÉTAPE 8 : Projeter les données sur les composants principaux
# = exprimer chaque point dans le nouvel espace réduit
# ──────────────────────────────────────────────
def projeter(data_centree, composants):
    projections = []                                           # liste des points projetés
    for row in data_centree:                                   # pour chaque point du dataset
        point = [sum(row[j] * composants[k][j]                # projection = produit scalaire avec le composant k
                     for j in range(len(row)))
                 for k in range(len(composants))]             # une coordonnée par composant
        projections.append(point)                             # ajouter le point projeté

    return projections                                         # retourner tous les points en 2D (ou nD)

# ──────────────────────────────────────────────
# ÉTAPE 9 : Calculer la variance expliquée par chaque composant
# = quelle proportion de l'info originale est conservée ?
# ──────────────────────────────────────────────
def variance_expliquee(valeurs):
    total = sum(valeurs)                                       # variance totale = somme de toutes les valeurs propres
    return [v / total * 100 for v in valeurs]                 # part de chaque composant en pourcentage

# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────
n_composants = 2                                              # on veut réduire à 2 dimensions

data_centree, moyennes = centrer(data)                        # étape 1 : centrer les données
print("Moyennes :", [f"{m:.2f}" for m in moyennes])           # afficher les moyennes de chaque colonne

cov = matrice_covariance(data_centree)                        # étape 2 : calculer la covariance
afficher_matrice(cov, "Matrice de Covariance")                # afficher la matrice

valeurs, composants = composants_principaux(cov, n_composants) # étape 3 : trouver les composants principaux

print("\n── Composants Principaux ──")
for i, (val, vect) in enumerate(zip(valeurs, composants)):    # pour chaque composant trouvé
    print(f"  PC{i+1} : valeur propre = {val:.2f} | direction = {[f'{v:.3f}' for v in vect]}")

var_exp = variance_expliquee(valeurs)                         # étape 4 : calculer la variance expliquée
print("\n── Variance Expliquée ──")
for i, v in enumerate(var_exp):                               # afficher le % de chaque composant
    print(f"  PC{i+1} : {v:.1f}%")
print(f"  Total conservé : {sum(var_exp):.1f}%")              # afficher le total conservé

projections = projeter(data_centree, composants)              # étape 5 : projeter les données en 2D
print("\n── Données Projetées (2D) ──")
print(f"  {'PC1':>10}  {'PC2':>10}")                          # entête du tableau
for i, p in enumerate(projections):                           # pour chaque point projeté
    print(f"  Point {i+1:2d} : {p[0]:10.2f}  {p[1]:10.2f}")  # afficher ses coordonnées PC1, PC2

Moyennes : ['23.60', '66.20', '18.00', '1017.90']

── Matrice de Covariance ──
       65.16      118.98      -53.00      -55.04
      118.98      224.40      -85.56     -101.31
      -53.00      -85.56       92.00       44.56
      -55.04     -101.31       44.56       46.99

── Composants Principaux ──
  PC1 : valeur propre = 376.61 | direction = ['0.413', '0.759', '-0.360', '-0.351']
  PC2 : valeur propre = 50.63 | direction = ['0.063', '0.374', '0.923', '-0.064']

── Variance Expliquée ──
  PC1 : 88.1%
  PC2 : 11.9%
  Total conservé : 100.0%

── Données Projetées (2D) ──
         PC1         PC2
  Point  1 :      29.14        2.87
  Point  2 :      18.41       14.59
  Point  3 :      10.40      -10.12
  Point  4 :      -5.85       -5.45
  Point  5 :     -22.67        4.02
  Point  6 :     -12.98       -2.96
  Point  7 :      18.07       -1.44
  Point  8 :       1.79       -5.91
  Point  9 :     -32.08        6.13
  Point 10 :      -4.24       -1.74
